# Zero-Shot Benchmark: OpenLID-v3
Baseline evaluation of the **unmodified** `HPLT/OpenLID-v3` FastText model on the preprocessed benchmark datasets.

This is the reference point for `06.finetune_models/finetune_OpenLID_expansion.ipynb`, which expands the OpenLID-v3 classification head with `san_Sinh` and `pli_Sinh`. The label space here is the same one used during fine-tuning, so the zero-shot and fine-tuned results can be compared directly.

Note: OpenLID-v3 has no `san_Sinh` or `pli_Sinh` label, so zero-shot recall on those classes is expected to be 0.

In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")

    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')

    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'

    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')

    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")

In [2]:
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'

## Step 1: Label Mapping & Evaluation Helpers
The mapping mirrors `DATASET_TO_OPENLID_MAP` from the fine-tuning notebook (without the `__label__` prefix). Sanskrit is split by script using the row's `source`: rows added by this project (DCS, SansinNT, SiDiaC-v2) are Sinhala script, the rest come from the original benchmark in Devanagari.

In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import re
import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Dataset 3-letter code -> OpenLID ISO3_Script label (same as finetune_OpenLID_expansion.ipynb)
DATASET_TO_OPENLID_MAP = {
    "eng": "eng_Latn",
    "sin": "sin_Sinh",
    # "san" is handled separately by script, see map_true_label()
    "pli": "pli_Sinh",  # Not in OpenLID-v3; added by the expansion fine-tune
    "tam": "tam_Taml",
    "hin": "hin_Deva",
    "ben": "ben_Beng",
    "arb": "ara_Arab",  # OpenLID uses 'ara' instead of 'arb'
    "fra": "fra_Latn",
    "deu": "deu_Latn",
    "jpn": "jpn_Jpan",
    "nld": "nld_Latn",
    "pol": "pol_Latn",
    "ita": "ita_Latn",
    "por": "por_Latn",
    "tur": "tur_Latn",
    "spa": "spa_Latn",
    "ell": "ell_Grek",
    "urd": "urd_Arab",
    "bul": "bul_Cyrl",
    "cmn": "cmn_Hans",
    "rus": "rus_Cyrl",
    "tha": "tha_Thai",
    "swh": "swh_Latn",
    "vie": "vie_Latn",
}

SINHALA_SANSKRIT_SOURCES = ["DCS", "SansinNT", "SiDiaC-v2"]

TARGET_LABELS = sorted(set(DATASET_TO_OPENLID_MAP.values()) | {"san_Sinh", "san_Deva"})


def map_true_label(row):
    label = str(row.get("label", "")).strip()
    source = str(row.get("source", "")).strip()

    if label == "san":
        # Sanskrit added by our project in Sinhala script
        if source in SINHALA_SANSKRIT_SOURCES:
            return "san_Sinh"
        # Sanskrit coming from the original benchmark
        return "san_Deva"

    return DATASET_TO_OPENLID_MAP.get(label)


def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")

    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            raw_label = row.get("label")
            if raw_label in DATASET_TO_OPENLID_MAP or raw_label == "san":
                records.append(row)

    df = pd.DataFrame(records)

    if df.empty:
        print("No matching target languages found in this dataset.")
        return df

    df["openlid_label"] = df.apply(map_true_label, axis=1)
    df = df[df["openlid_label"].notna()].copy()

    print(f"Loaded {len(df)} rows across {df['openlid_label'].nunique()} language-script classes")
    print("\nTrue-label counts:")
    print(df["openlid_label"].value_counts().sort_index())
    return df


def evaluate_and_save(results, model_name, dataset_name, target_labels):
    # Only score labels that actually occur in this dataset, so absent classes
    # don't drag macro F1 down.
    present_labels = [l for l in target_labels if l in set(results["true_label"])]

    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=present_labels, zero_division=0,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=present_labels, digits=4, zero_division=0,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return {"dataset": dataset_name, "n_samples": len(results), "accuracy": acc, "macro_f1": macro_f1}


dataset_files = sorted(glob.glob(os.path.join(input_dir, "*.jsonl")))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")
else:
    print("Datasets found:", [os.path.basename(f) for f in dataset_files])

Datasets found: ['commonlid.jsonl', 'flores_plus.jsonl', 'wili-2018.jsonl']


## Step 2: Load OpenLID-v3
Uses the same local path as the fine-tuning notebook (`models/pretrained/openlid/openlid-v3.bin`), downloading it from Hugging Face on first run.

In [4]:
import shutil
import fasttext
from huggingface_hub import hf_hub_download

model_dir = 'models/pretrained/openlid'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'openlid-v3.bin')

if not os.path.exists(model_path):
    print("Downloading OpenLID-v3 from Hugging Face...")
    downloaded_path = hf_hub_download(repo_id="HPLT/OpenLID-v3", filename="openlid-v3.bin")
    shutil.copy(downloaded_path, model_path)
    print(f"Model saved to {model_path}")
else:
    print(f"Model already exists at {model_path}")

# Suppress fasttext load warning
fasttext.FastText.eprint = lambda x: None
print("Loading OpenLID-v3 model...")
model = fasttext.load_model(model_path)
model_name = "OpenLID-v3"

model_labels = {l.replace("__label__", "") for l in model.get_labels()}
print(f"{model_name} supports {len(model_labels)} labels.")

missing = [l for l in TARGET_LABELS if l not in model_labels]
print(f"Target labels not in the model (zero-shot recall will be 0): {missing}")

openlid-v3.bin: reconstructing file:   0%|          |  0.00B / 1.23GB            

openlid-v3.bin: downloading bytes:           |  0.00B            

d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--HPLT--OpenLID-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model saved to models/pretrained/openlid\openlid-v3.bin
Loading OpenLID-v3 model...
OpenLID-v3 supports 195 labels.
Target labels not in the model (zero-shot recall will be 0): ['pli_Sinh', 'san_Sinh']


## Step 3: Run the Benchmark
Text is cleaned the same way as the OpenLID-v2 benchmark (lowercase, digits and punctuation stripped) so the two baselines are comparable.

In [5]:
def clean_for_openlid(text):
    text = str(text).strip().replace("\n", " ").lower()
    text = re.sub(r"[^\w\s]|\d", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

summary = []

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty:
        continue

    texts = df["text"].apply(clean_for_openlid).tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    preds, probs = model.predict(texts, k=1)

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["openlid_label"]
    results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
    results["confidence"] = [float(p[0]) for p in probs]

    summary.append(evaluate_and_save(results, model_name, dataset_name, TARGET_LABELS))


Loading commonlid.jsonl...
Loaded 142662 rows across 26 language-script classes

True-label counts:
openlid_label
ara_Arab    26152
ben_Beng     1886
bul_Cyrl      109
cmn_Hans      865
deu_Latn     7553
ell_Grek        7
eng_Latn    27461
fra_Latn     3233
hin_Deva     3666
ita_Latn     4387
jpn_Jpan     3344
nld_Latn     3299
pli_Sinh     3027
pol_Latn        1
por_Latn     2443
rus_Cyrl     4003
san_Deva      895
san_Sinh     1327
sin_Sinh     2693
spa_Latn     4236
swh_Latn    12383
tam_Taml       81
tha_Thai     3118
tur_Latn     4486
urd_Arab      204
vie_Latn    21803
Name: count, dtype: int64
Evaluating 142662 samples with OpenLID-v3...

ZERO-SHOT BENCHMARK RESULTS (OpenLID-v3 on commonlid)
Accuracy:  86.33%
Macro F1:  73.53%

Per-language breakdown:

              precision    recall  f1-score   support

    ara_Arab     0.9991    0.9901    0.9945     26152
    ben_Beng     0.9989    0.9470    0.9722      1886
    bul_Cyrl     0.7795    0.9083    0.8390       109
    cmn_Hans

## Step 4: Summary

In [6]:
summary_df = pd.DataFrame(summary)
summary_df["accuracy"] = (summary_df["accuracy"] * 100).round(2)
summary_df["macro_f1"] = (summary_df["macro_f1"] * 100).round(2)
print(f"{model_name} zero-shot summary (%):")
summary_df

OpenLID-v3 zero-shot summary (%):


,dataset,n_samples,accuracy,macro_f1
0,commonlid,142662,86.33,73.53
1,flores_plus,32347,73.84,80.38
2,wili-2018,27047,73.73,78.63
